In [1]:
# =========================================================
# FINAL RANK STACKING ENSEMBLE
# CatBoost + LightGBM + Logistic + ExtraTrees
# =========================================================

import numpy as np
import pandas as pd
import gc
import warnings
from itertools import combinations

from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

warnings.filterwarnings("ignore")

# =====================
# LOAD DATA
# =====================
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

TARGET = "Heart Disease"
train[TARGET] = train[TARGET].map({"Absence":0, "Presence":1}).astype(np.uint8)

y = train[TARGET].values
test_ids = test["id"]

X_tr_raw = train.drop(columns=[TARGET, "id"])
X_te_raw = test.drop(columns=["id"])

cat_cols = [
    'Sex','Chest pain type','FBS over 120','EKG results',
    'Exercise angina','Slope of ST',
    'Number of vessels fluro','Thallium'
]
num_cols = ['Age','BP','Cholesterol','Max HR','ST depression']

# =====================
# FREQUENCY ENCODING
# =====================
def freq_encode(tr, te, cols):
    tr_out, te_out = pd.DataFrame(index=tr.index), pd.DataFrame(index=te.index)
    for c in cols:
        freq = tr[c].value_counts(normalize=True)
        tr_out[c+"_freq"] = tr[c].map(freq)
        te_out[c+"_freq"] = te[c].map(freq)
    return tr_out, te_out

tr_freq, te_freq = freq_encode(X_tr_raw, X_te_raw, cat_cols + num_cols)

# =====================
# TARGET ENCODING
# =====================
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tr_te = pd.DataFrame(index=X_tr_raw.index)
te_te = pd.DataFrame(index=X_te_raw.index)

for c in cat_cols + num_cols:
    tr_te[c+"_te"] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = train.iloc[tr_i].groupby(c)[TARGET].mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(c+"_te")] = (
            X_tr_raw.iloc[val_i][c].map(means)
        )
    te_te[c+"_te"] = X_te_raw[c].map(train.groupby(c)[TARGET].mean())

# =====================
# FEATURE GROWTH
# =====================
base = tr_te.fillna(0)
corr_scores = {}

for a, b in combinations(base.columns, 2):
    corr_scores[(a,b)] = abs(np.corrcoef(base[a], base[b])[0,1])

top_pairs = sorted(corr_scores, key=corr_scores.get, reverse=True)[:8]

for a, b in top_pairs:
    tr_te[f"{a}_x_{b}"] = tr_te[a] * tr_te[b]
    te_te[f"{a}_x_{b}"] = te_te[a] * te_te[b]

X_train = pd.concat([tr_freq, tr_te], axis=1).fillna(0)
X_test  = pd.concat([te_freq, te_te], axis=1).fillna(0)

# =====================
# SCALER FOR LR
# =====================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# =====================
# MODELS
# =====================
cat = CatBoostClassifier(
    iterations=8000,
    learning_rate=0.01,
    depth=2,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    verbose=False
)

lgb_model = lgb.LGBMClassifier(
    objective="binary",
    learning_rate=0.01,
    num_leaves=4,
    max_depth=2,
    n_estimators=8000,
    n_jobs=-1
)

lr = LogisticRegression(
    penalty="elasticnet",
    C=0.2,
    l1_ratio=0.1,
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    n_jobs=-1
)

et = ExtraTreesClassifier(
    n_estimators=1200,
    max_depth=6,
    min_samples_leaf=20,
    max_features=0.7,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

# =====================
# CV TRAINING WITH RANK STACK
# =====================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_rank = np.zeros(len(X_train))
test_rank = np.zeros(len(X_test))

for fold, (tr, val) in enumerate(skf.split(X_train, y)):
    print(f"\nFold {fold+1}")

    # Fit models
    cat.fit(X_train.iloc[tr], y[tr])
    lgb_model.fit(X_train.iloc[tr], y[tr])
    lr.fit(X_train_scaled[tr], y[tr])
    et.fit(X_train.iloc[tr], y[tr])

    # Validation predictions
    val_preds = [
        cat.predict_proba(X_train.iloc[val])[:,1],
        lgb_model.predict_proba(X_train.iloc[val])[:,1],
        lr.predict_proba(X_train_scaled[val])[:,1],
        et.predict_proba(X_train.iloc[val])[:,1]
    ]

    # Test predictions
    test_preds = [
        cat.predict_proba(X_test)[:,1],
        lgb_model.predict_proba(X_test)[:,1],
        lr.predict_proba(X_test_scaled)[:,1],
        et.predict_proba(X_test)[:,1]
    ]

    # Convert to ranks
    val_rank = np.mean([rankdata(p) for p in val_preds], axis=0) / len(val)
    test_rank_fold = np.mean([rankdata(p) for p in test_preds], axis=0) / len(test)

    oof_rank[val] = val_rank
    test_rank += test_rank_fold / skf.n_splits

    print(" Fold AUC:", roc_auc_score(y[val], val_rank))
    gc.collect()

print("\nFinal CV AUC:", roc_auc_score(y, oof_rank))

# =====================
# SUBMISSION
# =====================
submission = pd.DataFrame({
    "id": test_ids,
    TARGET: test_rank
})

submission.to_csv("submission.csv", index=False)

print("\n🚀 RANK STACK SUBMISSION SAVED")



Fold 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.042336 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2762
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv